# Stormlight sentiment trajectory

Per-paragraph sentiment via VADER (lexicon-based, runs in seconds on the full corpus). Aggregated to per-chapter mean, then plotted as a line graph for each book:

- A thick black line shows the rolling-mean sentiment across all chapters (the overall narrative arc).
- Colored lines show individual top-POV character arcs — each character's line traces only their own chapters, in narrative order.

Scores are cached to `csv_data/paragraph_sentiment.csv`. Delete that file to force a re-score.


In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO        = Path("/Users/caputomachine/Desktop/StormlightCorpus")
PARAS_IN    = REPO / "csv_data" / "paragraphs.csv"
CHAPTERS_IN = REPO / "csv_data" / "chapters.csv"
CACHE_OUT   = REPO / "csv_data" / "paragraph_sentiment.csv"
PLOT_OUT    = "sentiment_trajectory.png"
HTML_OUT    = "sentiment_trajectory.html"

TOP_POVS_LINES = 6     # how many character lines to draw on the plot
SMOOTH_WIN     = 5     # chapters per rolling window for aggregate smoothing


In [ ]:
# ── 1. Score every paragraph with VADER (or load cache) ──────────────────────
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

def clean_for_sentiment(t: str) -> str:
    if not isinstance(t, str):
        return ""
    t = re.sub(r"\*+", "", t)        # strip italic/bold markers
    t = re.sub(r"-{2,}", " — ", t)    # collapse --- into a real em-dash
    return t

if CACHE_OUT.exists():
    paras = pd.read_csv(CACHE_OUT)
    print(f"loaded cached scores ({len(paras):,} paragraphs) from {CACHE_OUT.name}")
else:
    paras = pd.read_csv(PARAS_IN)
    print(f"running VADER on {len(paras):,} paragraphs ...")
    analyzer = SentimentIntensityAnalyzer()
    rows = [analyzer.polarity_scores(clean_for_sentiment(t)) for t in paras["text"].tolist()]
    scores = pd.DataFrame(rows)
    paras = pd.concat([paras.reset_index(drop=True), scores], axis=1)
    paras.to_csv(CACHE_OUT, index=False)
    print(f"wrote {CACHE_OUT.relative_to(REPO)}")

paras.head(3)[["book","heading_id","pov","compound","pos","neg","neu"]]


In [ ]:
# ── 2. Aggregate to one row per chapter, join chapter titles ─────────────────
ch = (
    paras.groupby(["book", "chapter_order", "heading_id", "pov"], as_index=False)
         .agg(n_paras=("text", "count"),
              compound_mean=("compound", "mean"),
              pos_mean=("pos", "mean"),
              neg_mean=("neg", "mean"))
)

titles = pd.read_csv(CHAPTERS_IN)[["book", "order", "heading_text"]].rename(columns={"order": "chapter_order"})
ch = ch.merge(titles, on=["book", "chapter_order"], how="left")
ch = ch.sort_values(["book", "chapter_order"]).reset_index(drop=True)

print(f"{len(ch)} sections scored.")
print(ch.head().to_string())


In [ ]:
# ── 3. Matplotlib plot — one subplot per book ────────────────────────────────
top_povs = ch["pov"].value_counts().head(TOP_POVS_LINES).index.tolist()
palette = plt.get_cmap("tab10", len(top_povs))
pov_color = {p: palette(i) for i, p in enumerate(top_povs)}

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharey=True)

for ax, book in zip(axes, ["The Way of Kings", "Words of Radiance"]):
    sub = ch[ch["book"] == book].sort_values("chapter_order")

    # Per-chapter raw sentiment (light scatter)
    ax.plot(sub["chapter_order"], sub["compound_mean"],
            color="lightgrey", marker="o", markersize=3,
            linewidth=0.6, alpha=0.55, zorder=2,
            label="All sections (raw)")

    # Aggregate rolling-mean line
    smoothed = sub["compound_mean"].rolling(SMOOTH_WIN, center=True, min_periods=1).mean()
    ax.plot(sub["chapter_order"], smoothed,
            color="black", linewidth=2.6, zorder=4,
            label=f"All sections ({SMOOTH_WIN}-chapter rolling mean)")

    # Per-POV arc — connect each character's chapters in order
    for pov in top_povs:
        pov_ch = sub[sub["pov"] == pov]
        if len(pov_ch) < 2:
            continue
        ax.plot(pov_ch["chapter_order"], pov_ch["compound_mean"],
                color=pov_color[pov], marker="o", markersize=5,
                linewidth=1.5, alpha=0.9, zorder=3,
                label=f"{pov}  (n={len(pov_ch)})")

    ax.axhline(0, color="grey", linestyle="--", linewidth=0.8, alpha=0.6, zorder=1)
    ax.set_title(f"{book}  —  VADER compound sentiment", fontsize=12, fontweight="bold")
    ax.set_ylabel("Sentiment (compound)")
    ax.grid(True, linestyle=":", alpha=0.35)
    ax.legend(loc="lower left", fontsize=8, ncol=2, framealpha=0.92)

axes[1].set_xlabel("Section order within book")
plt.tight_layout()
plt.savefig(PLOT_OUT, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {PLOT_OUT}")


## Interactive version

Same plot in Plotly so you can hover for chapter info and toggle character lines on/off in the legend.


In [ ]:
# ── 4. Plotly interactive — hover for chapter info ───────────────────────────
import plotly.graph_objects as go
import plotly.colors as pcol

qual = pcol.qualitative.D3
pov_color_hex = {p: qual[i % len(qual)] for i, p in enumerate(top_povs)}

fig = go.Figure()
book_axes = [("The Way of Kings", "x", "y"), ("Words of Radiance", "x2", "y2")]

for book, _, _ in book_axes:
    sub = ch[ch["book"] == book].sort_values("chapter_order")
    smoothed = sub["compound_mean"].rolling(SMOOTH_WIN, center=True, min_periods=1).mean()

    # Aggregate smoothed
    fig.add_trace(go.Scatter(
        x=sub["chapter_order"], y=smoothed,
        mode="lines", name=f"{book} — All ({SMOOTH_WIN}-ch smoothed)",
        line=dict(color="black", width=3), legendgroup=book,
        xaxis="x1" if book.startswith("The") else "x2",
        yaxis="y1" if book.startswith("The") else "y2",
    ))
    # Raw per-chapter
    fig.add_trace(go.Scatter(
        x=sub["chapter_order"], y=sub["compound_mean"],
        mode="markers", name=f"{book} — raw",
        marker=dict(color="lightgrey", size=5),
        customdata=np.stack([sub["heading_text"], sub["pov"], sub["n_paras"]], axis=-1),
        hovertemplate="<b>%{customdata[0]}</b><br>POV: %{customdata[1]}<br>"
                      "compound: %{y:.3f}<br>paragraphs: %{customdata[2]}<extra></extra>",
        legendgroup=book,
        xaxis="x1" if book.startswith("The") else "x2",
        yaxis="y1" if book.startswith("The") else "y2",
    ))
    # Per-character lines
    for pov in top_povs:
        pov_ch = sub[sub["pov"] == pov]
        if len(pov_ch) < 2:
            continue
        fig.add_trace(go.Scatter(
            x=pov_ch["chapter_order"], y=pov_ch["compound_mean"],
            mode="lines+markers", name=f"{book} — {pov}",
            line=dict(color=pov_color_hex[pov], width=1.6),
            marker=dict(size=6),
            customdata=np.stack([pov_ch["heading_text"], pov_ch["n_paras"]], axis=-1),
            hovertemplate=f"<b>%{{customdata[0]}}</b><br>POV: {pov}<br>"
                          "compound: %{y:.3f}<br>paragraphs: %{customdata[1]}<extra></extra>",
            legendgroup=book,
            xaxis="x1" if book.startswith("The") else "x2",
            yaxis="y1" if book.startswith("The") else "y2",
        ))

fig.update_layout(
    height=850, width=1200,
    title="Stormlight chapters — VADER sentiment trajectory",
    grid=dict(rows=2, columns=1, pattern="independent"),
    xaxis  = dict(title="The Way of Kings — section order"),
    yaxis  = dict(title="compound", zeroline=True, zerolinecolor="grey"),
    xaxis2 = dict(title="Words of Radiance — section order"),
    yaxis2 = dict(title="compound", zeroline=True, zerolinecolor="grey"),
    plot_bgcolor="white",
    hovermode="closest",
    legend=dict(font=dict(size=10), groupclick="toggleitem"),
)

fig.write_html(HTML_OUT, include_plotlyjs="cdn")
print(f"saved {HTML_OUT}")
fig.show()
